# Pipeline 1 - Donor Lapse Risk Predictor

Generated: 2026-04-06T22:13:08.300199Z


## CRISP-DM / Rubric Overview

Business understanding: Identify supporters most likely to lapse in the next 90 days so leaders can prioritize retention outreach.

Data understanding and preparation are implemented in code below. Each notebook includes a predictive model, an explanatory model, evaluation metrics, relationship-analysis notes, and JSON export for app deployment.


In [ ]:
import json
from pathlib import Path
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import GradientBoostingClassifier, GradientBoostingRegressor, RandomForestClassifier, RandomForestRegressor
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.metrics import accuracy_score, mean_absolute_error, mean_squared_error, precision_recall_fscore_support, r2_score, roc_auc_score, top_k_accuracy_score
from sklearn.model_selection import KFold, RandomizedSearchCV, StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

cwd = Path.cwd().resolve()
REPO_ROOT = cwd.parent if cwd.name == "ml-pipelines" else cwd
DATA_DIR = REPO_ROOT / "data" / "raw"
OUT_DIR = REPO_ROOT / "output" / "ml-predictions"
OUT_DIR.mkdir(parents=True, exist_ok=True)
print("Data dir:", DATA_DIR)
print("Output dir:", OUT_DIR)

def require_csv(stem):
    path = DATA_DIR / f"{stem}.csv"
    if not path.exists():
        raise FileNotFoundError(f"Missing {path}. Put the INTEX CSVs under data/raw/.")
    return pd.read_csv(path, encoding="utf-8-sig")

def make_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=False)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=False)

def as_bool(series):
    if series.dtype == bool:
        return series.fillna(False)
    return series.astype(str).str.lower().isin(["true", "1", "yes", "y"])

def numeric(series, fill_value=0.0):
    return pd.to_numeric(series, errors="coerce").fillna(fill_value)

def fill_numeric_median(df, cols):
    for col in cols:
        vals = pd.to_numeric(df[col], errors="coerce")
        df[col] = vals.fillna(vals.median() if vals.notna().any() else 0.0)

def rmse(y_true, y_pred):
    return float(np.sqrt(mean_squared_error(y_true, y_pred)))

def eval_classification(y_true, y_pred, y_proba=None):
    acc = accuracy_score(y_true, y_pred)
    pr, rc, f1, _ = precision_recall_fscore_support(y_true, y_pred, average="binary", zero_division=0)
    out = {"accuracy": float(acc), "precision": float(pr), "recall": float(rc), "f1": float(f1)}
    if y_proba is not None and pd.Series(y_true).nunique() > 1:
        try:
            out["roc_auc"] = float(roc_auc_score(y_true, y_proba))
        except Exception:
            pass
    return out

def eval_regression(y_true, y_pred):
    return {"mae": float(mean_absolute_error(y_true, y_pred)), "rmse": rmse(y_true, y_pred), "r2": float(r2_score(y_true, y_pred))}

def classification_baseline(y_train, y_test):
    majority = pd.Series(y_train).mode().iloc[0]
    pred = pd.Series([majority] * len(y_test), index=pd.Series(y_test).index)
    return {"majority_class": str(majority), "accuracy": float(accuracy_score(y_test, pred))}

def regression_baseline(y_train, y_test):
    median_value = float(pd.Series(y_train).median())
    pred = np.repeat(median_value, len(y_test))
    out = eval_regression(y_test, pred)
    out["baseline_value"] = median_value
    return out

def top_features(pipe, n=10):
    if "pre" not in pipe.named_steps:
        return pd.DataFrame()
    model = pipe.named_steps.get("model") or pipe.named_steps.get("m")
    if model is None:
        return pd.DataFrame()
    try:
        names = pipe.named_steps["pre"].get_feature_names_out()
    except Exception:
        return pd.DataFrame()
    if hasattr(model, "feature_importances_"):
        weights = np.asarray(model.feature_importances_)
    elif hasattr(model, "coef_"):
        weights = np.abs(np.asarray(model.coef_)).mean(axis=0) if np.asarray(model.coef_).ndim > 1 else np.abs(np.asarray(model.coef_))
    else:
        return pd.DataFrame()
    out = pd.DataFrame({"feature": names, "importance": weights}).sort_values("importance", ascending=False).head(n)
    return out.reset_index(drop=True)

def print_business_takeaway(text):
    print("\nBusiness takeaway:")
    print(text)

def compact_cv_classification(X, y, cat_cols, num_cols, scoring_metric="roc_auc", cv_splits=3):
    y = pd.Series(y)
    min_class = y.value_counts().min()
    if y.nunique() < 2 or min_class < 2:
        print("Skipping CV comparison: target does not have enough samples in each class.")
        return pd.DataFrame()
    n_splits = int(min(cv_splits, min_class))
    cv = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    models = {
        "Logistic Regression": LogisticRegression(max_iter=3000),
        "Random Forest": RandomForestClassifier(n_estimators=150, random_state=42, min_samples_leaf=3),
        "Gradient Boosting": GradientBoostingClassifier(random_state=42),
    }
    rows = []
    scoring = {"accuracy": "accuracy", "f1_weighted": "f1_weighted"}
    if y.nunique() == 2:
        scoring["roc_auc"] = "roc_auc"
    for name, model in models.items():
        pipe = Pipeline([("pre", prep(cat_cols, num_cols)), ("model", model)])
        scores = cross_validate(pipe, X, y, cv=cv, scoring=scoring, n_jobs=1, error_score=np.nan)
        row = {"Model": name}
        for metric in scoring:
            vals = scores[f"test_{metric}"]
            row[f"{metric}_mean"] = float(np.nanmean(vals))
            row[f"{metric}_std"] = float(np.nanstd(vals))
        rows.append(row)
    out = pd.DataFrame(rows).sort_values(f"{'roc_auc' if 'roc_auc' in scoring else 'accuracy'}_mean", ascending=False)
    print("Compact cross-validation comparison:")
    print(out.to_string(index=False))
    return out

def compact_holdout_regression(train, test, features, target, cat_cols, num_cols, log_target=False):
    models = {
        "Linear Regression": LinearRegression(),
        "Random Forest": RandomForestRegressor(n_estimators=150, random_state=42, min_samples_leaf=3),
        "Gradient Boosting": GradientBoostingRegressor(random_state=42),
    }
    rows = []
    best = None
    best_mae = float("inf")
    y_train = np.log1p(train[target]) if log_target else train[target]
    for name, model in models.items():
        pipe = Pipeline([("pre", prep(cat_cols, num_cols)), ("model", model)])
        pipe.fit(train[features], y_train)
        pred = pipe.predict(test[features])
        pred = np.expm1(pred) if log_target else pred
        pred = np.maximum(0, pred)
        metrics = eval_regression(test[target], pred)
        rows.append({"Model": name, **metrics})
        if metrics["mae"] < best_mae:
            best_mae = metrics["mae"]
            best = (name, pipe)
    out = pd.DataFrame(rows).sort_values("mae")
    print("Time/holdout model comparison:")
    print(out.to_string(index=False))
    return out, best

def compact_randomized_tune_regressor(train, features, target, cat_cols, num_cols, log_target=False, n_iter=6):
    y_train = np.log1p(train[target]) if log_target else train[target]
    cv = KFold(n_splits=min(3, max(2, len(train) // 20)), shuffle=True, random_state=42)
    pipe = Pipeline([("pre", prep(cat_cols, num_cols)), ("model", RandomForestRegressor(random_state=42))])
    search = RandomizedSearchCV(
        pipe,
        {
            "model__n_estimators": [100, 150, 250],
            "model__min_samples_leaf": [1, 3, 5, 8],
            "model__max_depth": [None, 3, 5, 8],
        },
        n_iter=n_iter,
        scoring="neg_mean_absolute_error",
        cv=cv,
        random_state=42,
        n_jobs=1,
    )
    search.fit(train[features], y_train)
    print("RandomizedSearchCV best params:", search.best_params_)
    print("RandomizedSearchCV best CV MAE:", float(-search.best_score_))
    return search.best_estimator_

def time_split(df, date_col, test_frac=0.25):
    df = df.sort_values(date_col).copy()
    split_idx = max(1, int(len(df) * (1 - test_frac)))
    split_idx = min(split_idx, len(df) - 1)
    return df.iloc[:split_idx].copy(), df.iloc[split_idx:].copy()

def safe_classifier_split(X, y, test_size=0.25):
    stratify = y if y.nunique() > 1 and y.value_counts().min() >= 2 else None
    return train_test_split(X, y, test_size=test_size, random_state=42, stratify=stratify)

def score_bands(scores):
    scores = pd.Series(scores).astype(float)
    if len(scores) == 0 or scores.nunique(dropna=True) < 2:
        return pd.Series(["Medium"] * len(scores), index=scores.index)
    labels = ["Low", "Medium", "High", "Very High"]
    q = min(4, scores.nunique(dropna=True), len(scores))
    try:
        return pd.qcut(scores.rank(method="first"), q=q, labels=labels[:q], duplicates="drop").astype(str)
    except Exception:
        return pd.Series(["Medium"] * len(scores), index=scores.index)

def prep(cat_cols, num_cols):
    return ColumnTransformer([("cat", make_encoder(), cat_cols), ("num", "passthrough", num_cols)])

def export_predictions_json(prediction_type, entity_type, df_out, id_col, score_col, label_col=None):
    out_path = OUT_DIR / f"{prediction_type}.json"
    excluded = {id_col, score_col}
    if label_col:
        excluded.add(label_col)
    rows = []
    for _, row in df_out.iterrows():
        payload = {k: v for k, v in row.items() if k not in excluded}
        rows.append({
            "predictionType": prediction_type,
            "entityType": entity_type,
            "entityId": int(row[id_col]),
            "score": float(row[score_col]),
            "label": None if label_col is None or pd.isna(row[label_col]) else str(row[label_col]),
            "payloadJson": json.dumps(payload, default=str),
        })
    out_path.write_text(json.dumps(rows, indent=2), encoding="utf-8")
    print(f"Wrote {len(rows)} predictions:", out_path)


In [ ]:
supporters = require_csv("supporters")
donations = require_csv("donations")
donations["donation_date"] = pd.to_datetime(donations["donation_date"], errors="coerce")
donations = donations.dropna(subset=["donation_date", "supporter_id"]).copy()
donations["amount"] = numeric(donations["amount"])
donations["estimated_value"] = numeric(donations["estimated_value"])
donations["is_recurring_bool"] = as_bool(donations["is_recurring"])

cutoff = donations["donation_date"].max() - pd.Timedelta(days=90)
label_end = cutoff + pd.Timedelta(days=90)
past = donations[donations["donation_date"] <= cutoff].copy()
future = donations[(donations["donation_date"] > cutoff) & (donations["donation_date"] <= label_end)].copy()

base = supporters[["supporter_id","supporter_type","relationship_type","region","country","status","acquisition_channel","first_donation_date"]].copy()
base["first_donation_date"] = pd.to_datetime(base["first_donation_date"], errors="coerce")
agg = past.groupby("supporter_id").agg(
    donation_count=("donation_id","count"),
    last_donation_date=("donation_date","max"),
    distinct_campaigns=("campaign_name", lambda s: s.dropna().nunique()),
    distinct_channels=("channel_source", lambda s: s.dropna().nunique()),
    recurring_any=("is_recurring_bool","max"),
    total_value=("estimated_value","sum"),
).reset_index()
mon = past[past["donation_type"] == "Monetary"]
mon_agg = mon.groupby("supporter_id").agg(
    monetary_count=("donation_id","count"),
    monetary_sum=("amount","sum"),
    monetary_avg=("amount","mean"),
    monetary_max=("amount","max"),
).reset_index()
future_gave = (future.groupby("supporter_id")["donation_id"].count() > 0).astype(int).rename("gave_next_90d").reset_index()

df = base.merge(agg, on="supporter_id", how="left").merge(mon_agg, on="supporter_id", how="left").merge(future_gave, on="supporter_id", how="left")
df = df[df["donation_count"].fillna(0) > 0].copy()
df["gave_next_90d"] = df["gave_next_90d"].fillna(0).astype(int)
df["lapsed_next_90d"] = 1 - df["gave_next_90d"]
df["recency_days"] = (cutoff - df["last_donation_date"]).dt.days.clip(lower=0)
df["donor_age_days"] = (cutoff - df["first_donation_date"]).dt.days.clip(lower=0)
cat_cols = ["supporter_type","relationship_type","region","country","status","acquisition_channel"]
num_cols = ["donation_count","distinct_campaigns","distinct_channels","recurring_any","total_value","monetary_count","monetary_sum","monetary_avg","monetary_max","recency_days","donor_age_days"]
df[cat_cols] = df[cat_cols].fillna("Unknown")
fill_numeric_median(df, num_cols)
print("Rows:", len(df), "lapse rate:", round(df["lapsed_next_90d"].mean(), 3))


In [ ]:
features = cat_cols + num_cols
X_train, X_test, y_train, y_test = safe_classifier_split(df[features], df["lapsed_next_90d"])
print("Baseline:", classification_baseline(y_train, y_test))
compact_cv_classification(df[features], df["lapsed_next_90d"], cat_cols, num_cols)
predictive = Pipeline([("pre", prep(cat_cols, num_cols)), ("model", GradientBoostingClassifier(random_state=42))])
predictive.fit(X_train, y_train)
proba = predictive.predict_proba(X_test)[:, 1]
print("Predictive:", eval_classification(y_test, (proba >= 0.5).astype(int), proba))
print("Top predictive features:")
print(top_features(predictive).to_string(index=False))

explanatory = Pipeline([("pre", prep(cat_cols, num_cols)), ("model", LogisticRegression(max_iter=3000))])
explanatory.fit(X_train, y_train)
proba_exp = explanatory.predict_proba(X_test)[:, 1]
print("Explanatory:", eval_classification(y_test, (proba_exp >= 0.5).astype(int), proba_exp))
print("Top explanatory relationships:")
print(top_features(explanatory).to_string(index=False))
print_business_takeaway("Prioritize high-risk supporters for retention outreach, especially when recency and low engagement patterns suggest they may lapse.")


In [ ]:
df_out = df[["supporter_id"] + features].copy()
df_out["risk_score"] = predictive.predict_proba(df_out[features])[:, 1]
df_out["risk_band"] = score_bands(df_out["risk_score"])
export_predictions_json(
    "donor_lapse_90d",
    "Supporter",
    df_out[["supporter_id","risk_score","risk_band","recency_days","donation_count","monetary_sum","recurring_any","acquisition_channel","supporter_type"]],
    "supporter_id",
    "risk_score",
    "risk_band",
)


## Deployment

Import the exported JSON with `POST /api/ml/import?replace=true` and view results in `/app/ml`.
